In [1]:
import pandas as pd
import numpy as np
from scipy import sparse as sps

%matplotlib inline

In [2]:
class User: 
    
    def __init__(self, uid, indices, values): 
        self._uid = uid
        self._indices = indices
        self._values = values
        
    @property          
    def uid(self):
        return  self._uid
        
    @property          
    def indices(self):
        return  self._indices
    
    @property          
    def values(self):
        return self._values
    
    def __repr__(self): 
        return str(f'User {self.uid}')

In [3]:
class EASE:
    
    def __init__(self, reg_weight=100):
        
        self.reg_weight = reg_weight
        self.B = None
        self.n = None
        
    def fit(self, X):

        G = X.T @ X
        G += self.reg_weight * sps.identity(G.shape[0])
        G = G.todense()
        P = np.linalg.inv(G)
        B = P / (-np.diag(P))
        np.fill_diagonal(B, 0.)
        self.B = B
        self.n = self.B.shape[0]
        
    def predict(self, user: User, top=10): 
        
        x_user = np.zeros(self.n)
        x_user[user.indices] = user.values

        prediction = np.array(x_user.dot(self.B))[0]

        prediction[user.indices] = -np.inf
        result = np.argpartition(prediction, -top)[-top:] 
        
        return result[::-1]

### Cохранение модели

In [4]:
import pickle

In [5]:
ease_model = pickle.load(open('EASE.pickle','rb'))

In [7]:
fid2efid = pickle.load(open('dict_fid2efid.pickle', 'rb'))

In [8]:
user = User(1, [893361, 470689, 4321512, 505827, 840821], [7, 8, 5, 4, 5])

In [9]:
# user = User(i, [1, 2, 3, 4, 5], [7, 8, 5, 4, 5])

In [10]:
class Predictor: 
    
    def __init__(self, model, fid2efid):
        
        self.fid2efid = fid2efid
        self.model = model 
        self.efid2fid = {v:k for k,v in fid2efid.items()}
        
        
    def transform(self, user):

        transformed = [(self.fid2efid[idx], val) 
                       for idx, val in zip(user.indices, user.values) 
                       if idx in fid2efid]

        new_indices, new_values = map(list, zip(*transformed)) if transformed else ([], [])

        return User(user.uid, list(new_indices), list(new_values)) 

        
    def predict(self, user, top=10):
        
        user = self.transform(user)
        
        return list(map(lambda x: self.efid2fid[x], self.model.predict(user, top=top)))  if user.indices else []    

In [11]:
predictor = Predictor(ease_model, fid2efid)

In [12]:
predictor.predict(user)

[841081,
 909898,
 718811,
 505898,
 928391,
 1048602,
 648440,
 662359,
 1322324,
 1348487]

Хороший, плохой, злой (1966) - 1007

На несколько долларов больше (1965) - 2519

Брат (1997) - 1803

Война (2002) - 2102

Перевозчик (2002) - 1178

In [14]:
user = User(1, [1007, 2519, 1803, 2102, 1178], [8, 9, 9, 9, 8])

In [16]:
ease_model.predict(user)

array([ 473, 1684,  855, 1132,  882, 1414,  880,  900, 2129, 1931])

In [20]:
efid2name = pickle.load(open('efid2name.pickle', 'rb'))

In [21]:
for i in ease_model.predict(user): 
    print(efid2name[i])

Брат 2 (2000)
Жмурки (2005)
Матрица (1999)
Лицо со шрамом (1983)
Бешеные псы (1991)
Дурак (2014)
Карты, деньги, два ствола (1998)
Кавказская пленница, или Новые приключения Шурика (1966)
Гран Торино (2008)
Сталкер (1979)


### Второй пример 

Трое из Простоквашино (1978) - 2368

Гарри Поттер и философский камень (2001) - 1186

Зверополис (2016) - 1244

Ну, погоди! (сериал, 1969 – 2017) - 2511

Ледниковый период (2002) - 1013

In [22]:
user = User(1, [2368, 1186, 1244, 2511, 1013], [8, 9, 9, 9, 8])

In [23]:
ease_model.predict(user)

array([1565,  132,  394,  970, 1457, 1187,   85,  744, 1818, 1406])

In [25]:
for i in ease_model.predict(user): 
    print(efid2name[i])

Легенда №17 (2012)
Головоломка (2015)
Шрэк (2001)
Гарри Поттер и Кубок огня (2005)
Жил-был пёс (1982)
Гарри Поттер и Тайная комната (2002)
Дэдпул (2016)
Гарри Поттер и узник Азкабана (2004)
Мадагаскар (2005)
Гарри Поттер и Дары Смерти: Часть II (2011)
